# Session 5: Chatbot, Streaming, Pydantic, Async + Neural Networks

**LLM Engineering Masterclass -- GrowAI**

---

## Setup: Connect to Ollama (Local AI)

Ollama runs AI models on your own laptop — free, private, no API key needed.

In [ ]:
!pip install ollama -q

In [ ]:
import ollama

# Ollama runs locally -- no API key needed!
# Make sure Ollama is running: open terminal, type: ollama serve
# Then pull a model: ollama pull llama3.2
print("Ollama library loaded!")

In [ ]:
# Quick test: is Ollama running?
try:
    models = ollama.list()
    print("Ollama is running!")
    print("Available models:", [m.model for m in models.models])
except Exception as e:
    print(f"Ollama not running! Start it with: ollama serve")
    print(f"Error: {e}")

In [ ]:
def ask_ai(prompt):
    response = ollama.chat(
        model="llama3.2",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Keep responses to 2-3 sentences."},
            {"role": "user", "content": prompt}
        ]
    )
    return response.message.content

In [ ]:
answer = ask_ai("What is Python in one sentence?")
print(answer)

---

## Part 1: Chatbot with Memory

Problem: ask_ai() has no memory. Each call is independent.

In [ ]:
# No memory -- model doesn't know what 'there' means
print(ask_ai("What is the capital of France?"))

In [ ]:
print(ask_ai("What language do they speak there?"))

Solution: send the ENTIRE conversation history every time.

### How Memory Works (Diagram)


**Without memory:** Each call is independent. Model forgets.
**With memory:** Full conversation history sent every time. Model reads all of it.

In [ ]:
def chat():
    messages = [
        {"role": "system", "content": "You are a friendly AI assistant. Keep responses to 2-3 sentences."}
    ]

    print("Chatbot ready! Type 'quit' to exit.\n")

    while True:
        user_input = input("You: ")

        if user_input.lower() == "quit":
            print("Goodbye!")
            break

        messages.append({"role": "user", "content": user_input})

        response = ollama.chat(
            model="llama3.2",
            messages=messages
        )

        assistant_message = response.message.content
        messages.append({"role": "assistant", "content": assistant_message})

        print(f"AI: {assistant_message}\n")

In [ ]:
chat()

---

## Part 2: Streaming

stream=True makes words appear one by one instead of waiting for the full answer.

### Streaming Explained (Diagram)


**Without streaming:** Wait... wait... BOOM full answer.
**With streaming:** Words flow in live, one by one.

In [ ]:
def ask_ai_stream(prompt):
    stream = ollama.chat(
        model="llama3.2",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        stream=True
    )

    full_response = ""
    for chunk in stream:
        word = chunk.message.content
        print(word, end="", flush=True)
        full_response += word

    print()
    return full_response

In [ ]:
# Without streaming
print("--- Without streaming ---")
answer = ask_ai("Tell me a fun fact about space")
print(answer)

In [ ]:
# With streaming -- watch words appear one by one
print("--- With streaming ---")
answer = ask_ai_stream("Tell me a fun fact about the ocean")

---

## Part 3: Pydantic -- Validate AI Outputs

Define what data SHOULD look like. Reject anything that doesn't match.

### Pydantic Flow (Diagram)


Messy AI output → Pydantic validates → Clean data or clear rejection.

In [ ]:
from pydantic import BaseModel

In [ ]:
class Person(BaseModel):
    name: str
    age: int
    city: str

In [ ]:
# Valid data
person1 = Person(name="Priya", age=24, city="Mumbai")
print(person1)
print(f"Name: {person1.name}")

In [ ]:
# Invalid data -- REJECTED
try:
    bad_person = Person(name="Rahul", age="twenty-five", city="Delhi")
except Exception as e:
    print(f"REJECTED: {e}")

In [ ]:
# Missing field -- REJECTED
try:
    incomplete = Person(name="Arun", age=30)
except Exception as e:
    print(f"REJECTED: {e}")

In [ ]:
import json

# Ask AI for structured data
response = ask_ai(
    "Give me a fictional person with name, age, and city. "
    "Respond ONLY in JSON format like: {\"name\": \"...\", \"age\": ..., \"city\": \"...\"}"
)
print("Raw AI response:", response)

In [ ]:
# Validate AI response
try:
    data = json.loads(response)
    person = Person(**data)
    print(f"VALID! Name: {person.name}, Age: {person.age}, City: {person.city}")
except json.JSONDecodeError:
    print("AI didn't return valid JSON!")
except Exception as e:
    print(f"Validation failed: {e}")

---

## Part 4: Async -- Parallel AI Calls

### Sequential vs Parallel (Diagram)


**Sequential:** 5 + 5 + 5 = 15 seconds.
**Parallel:** All at once = 5 seconds.

In [ ]:
import time

questions = [
    "What is gravity in one sentence?",
    "What is photosynthesis in one sentence?",
    "What is democracy in one sentence?"
]

In [ ]:
# SEQUENTIAL: one by one
start = time.time()
answers = []
for q in questions:
    answers.append(ask_ai(q))
end = time.time()

print(f"Sequential: {end - start:.1f} seconds")
for i, a in enumerate(answers):
    print(f"  Q{i+1}: {a}")

In [ ]:
import asyncio

# PARALLEL: all at once
async def ask_ai_async(prompt):
    return await asyncio.to_thread(ask_ai, prompt)

async def ask_all():
    tasks = [ask_ai_async(q) for q in questions]
    return await asyncio.gather(*tasks)

start = time.time()
answers = await ask_all()
end = time.time()

print(f"Parallel: {end - start:.1f} seconds")
for i, a in enumerate(answers):
    print(f"  Q{i+1}: {a}")

---

## Part 5: File I/O

In [ ]:
# Write a text file
with open("hello.txt", "w") as f:
    f.write("This is my first file from Python!")
print("Written!")

In [ ]:
# Read it back
with open("hello.txt", "r") as f:
    content = f.read()
print(content)

In [ ]:
import json

# Write a config as JSON
config = {
    "model": "llama3.2",
    "system_prompt": "You are a helpful coding tutor.",
    "temperature": 0.7
}
with open("config.json", "w") as f:
    json.dump(config, f, indent=2)
print("Config saved!")

In [ ]:
# Read JSON back
with open("config.json", "r") as f:
    cfg = json.load(f)
print(cfg["model"])
print(cfg["system_prompt"])

In [ ]:
# Use config to control AI
response = ollama.chat(
    model=cfg["model"],
    messages=[
        {"role": "system", "content": cfg["system_prompt"]},
        {"role": "user", "content": "How do I write a for loop?"}
    ]
)
print(response.message.content)

---

## Part 6: Virtual Environments (reference only)

In [ ]:
print("""
VIRTUAL ENVIRONMENT COMMANDS (for your terminal):

1. Create:    python -m venv myproject_env
2. Activate:  source myproject_env/bin/activate   (Mac/Linux)
              myproject_env\\Scripts\\activate      (Windows)
3. Install:   pip install ollama pydantic
4. Check:     pip freeze
5. Exit:      deactivate

To run Ollama:
  1. Download from https://ollama.com
  2. Terminal: ollama serve
  3. Terminal: ollama pull llama3.2
""")

### Neural Network Diagrams

#### The Training Loop

Forward → Loss → Backward → Update → Repeat millions of times.

#### Self-Attention (Transformer)

"The cat sat on the mat because **it** was tired" — 'it' pays strong attention to 'cat'.

#### Progression

Neuron → Layer → Network → **TRANSFORMER** (next session)

---

## Part 7: Neural Networks (whiteboard only, no code)

See Excalidraw diagrams for:
- Single neuron (inputs, weights, activation, output)
- Layers and network (deep = many layers)
- Training loop (forward > loss > backward > update > repeat)
- Self-attention (transformer teaser)

---

## Homework

1. Streaming chatbot with creative system prompt
2. Pydantic validation (Movie/Recipe/Book model)
3. (Optional) Create venv on local machine

## Next Session: Transformer deep dive + HuggingFace